# Arena 3D Gaussian Splatting - Google Colab

93 images of a 10-15m arena reconstructed using Gaussian Splatting.

## Instructions
1. Run cells in order
2. Upload resized images as ZIP when prompted
3. Download point cloud after training

---

In [ ]:
#@title Step 1: Install & Clone
%cd /content

# Install system packages
!apt-get update
!apt-get install -y colmap wget unzip

# Install Python deps
!pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118
!pip install plyfile numpy pillow opencv-python-headless tqdm

# Clone 3DGS with submodules
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting
%cd /content/gaussian-splatting

# Install rasterization libraries
!pip install submodules/diff-gaussian-rasterization
!pip install submodules/simple-knn

# Verify
!python -c "import diff_gaussian_rasterization; print('OK')"

In [ ]:
#@title Step 2: Upload Images
from google.colab import files
import zipfile
import os

!mkdir -p /content/gaussian-splatting/input

print("Upload a ZIP file containing your images")
uploaded = files.upload()

for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/gaussian-splatting/input')
        print(f"Extracted {fname}")
        break

# Count
imgs = [f for f in os.listdir('/content/gaussian-splatting/input') if f.lower().endswith(('.jpg', '.png'))]
print(f"{len(imgs)} images ready")

In [ ]:
#@title Step 3: COLMAP
%cd /content/gaussian-splatting

!mkdir -p sparse

# Feature extraction
!colmap feature_extractor \
    --database_path sparse/database.db \
    --image_path input

# Matching
!colmap exhaustive_matcher \
    --database_path sparse/database.db

# Mapping
!colmap mapper \
    --database_path sparse/database.db \
    --image_path input \
    --output_path sparse

!ls sparse/0/

In [ ]:
#@title Step 4: Convert
%cd /content/gaussian-splatting

!mkdir -p output/train

!python convert.py \
    --source_path input \
    --model_path sparse/0 \
    --output_path output/train

!ls output/train/

In [ ]:
#@title Step 5: Train
%cd /content/gaussian-splatting

!python train.py \
    -s output/train \
    -i 0 \
    --iterations 3000

In [ ]:
#@title Step 6: Get Point Cloud
import os

# Find latest iteration
pc_dir = "output/train/point_cloud"
iters = [d for d in os.listdir(pc_dir) if d.startswith('iteration_')]
if iters:
    latest = sorted(iters)[-1]
    pc_path = os.path.join(pc_dir, latest)
    
    # Find ply file
    ply = [f for f in os.listdir(pc_path) if f.endswith('.ply')]
    if ply:
        dest = f"/content/arena_{latest}.ply"
        !cp {os.path.join(pc_path, ply[0])} {dest}
        print(f"Saved: {dest}")
        !ls -lh {dest}

In [ ]:
#@title Step 7: Download
from google.colab import files

# Download
files.download('/content/arena_iteration_3000.ply')